# Project 2 · Audit of five transaction-cleaning rules

Run the cells in order on a CPU. They read the raw workbook, count rows affected by five rules, and write a single audit table. The table describes the first stage of cleaning. Later stages handle merchandise codes and aggregate purchases into baskets.

# 项目二 · 五项交易清洗规则审计

使用 CPU，按顺序运行下方单元格。代码读取原始工作簿，统计五项规则影响的行数，并写出一张审计表。该表记录清洗的第一阶段。后续阶段另行处理非商品编码并聚合购物篮。

## 1. Select the workbook and output path

In Colab, authorize Drive access when prompted. The default project folder in My Drive is `CS5481_Project2_OnlineRetail`. Place the workbook under `data/raw/`. The CSV goes in `results/`. If prompted, copy the workbook path from Colab's Files panel. If you select a file outside the project's `data/raw/` layout, also enter the full path of the Drive project folder where you want to save the audit. Locally, start Jupyter in the repository root or its `notebooks/` directory. Check the two displayed paths before running the audit.

## 1. 选择工作簿和输出路径

在 Colab 中按提示授权访问 Drive。「我的云端硬盘」下的默认项目文件夹为 `CS5481_Project2_OnlineRetail`。工作簿放在 `data/raw/`，审计表写入 `results/`。如果出现路径输入框，请在 Colab 左侧「文件」面板中复制工作簿路径。如果选取的文件不在项目的 `data/raw/` 目录下，还需输入用于保存结果的 Drive 项目文件夹完整路径。本地运行时，从仓库根目录或 `notebooks/` 启动 Jupyter。运行审计前，请核对显示的输入和输出路径。

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail")
    workbook = project / "data/raw/Online Retail.xlsx"
    if not workbook.is_file():
        workbook = Path(input("Workbook path / 工作簿完整路径：").strip())
        if not workbook.is_file():
            raise FileNotFoundError("Workbook not found / 找不到工作簿")
        if workbook.parent.name == "raw" and workbook.parent.parent.name == "data":
            project = workbook.parents[2]
        else:
            project = Path(input("Drive project folder for results / 保存结果的 Drive 项目文件夹完整路径：").strip())
            if not project.is_absolute() or not project.is_dir():
                raise ValueError("Select an existing project folder / 请选择已存在的项目文件夹")
else:
    project = Path.cwd()
    if project.name == "notebooks":
        project = project.parent
    workbook = project / "data/raw/Online Retail.xlsx"

if not workbook.is_file():
    raise FileNotFoundError(f"Workbook not found / 找不到工作簿：{workbook}")

output = project / "results/cleaning_audit.csv"
print("Workbook / 工作簿：", workbook)
print("Audit output / 审计表：", output)

## 2. Load the audit implementation

The notebook calls `run_audit` from `src/data/clean.py`. Locally, it uses the source in the current checkout. Colab fetches the matching file from a specific Git commit and checks its SHA-256 before loading it. If the download fails, open the source link printed below, sign in to GitHub if needed, download the raw file, and select it in the upload box. The same checksum check applies to that file.

## 2. 加载审计实现

Notebook 调用 `src/data/clean.py` 中的 `run_audit`。本地运行时使用当前仓库的源码。Colab 从指定的 Git 提交读取对应文件，核对其 SHA-256 后再加载。如果下载失败，请打开下方打印的源码链接，按需登录 GitHub，下载原始文件并在上传框中选中它。手动选择的文件也会经过相同的校验。

In [ ]:
if drive is None:
    import sys

    sys.path.insert(0, str(project))
    from src.data.clean import run_audit
else:
    from hashlib import sha256
    from runpy import run_path
    from tempfile import TemporaryDirectory
    from urllib.request import urlopen

    repository = "NobunagaK/online-retail-top10-recommendation"
    source_commit = "48b08c2fd60e969268bcf5d55e18db4ea593fe4f"
    expected_digest = "b30a804762918b0707fb0b8bede67edfc9c67ae5f7849d85f2f0dca954f9a1b4"
    source_path = "src/data/clean.py"
    source_url = f"https://raw.githubusercontent.com/{repository}/{source_commit}/{source_path}"
    try:
        with urlopen(source_url, timeout=20) as response:
            source = response.read()
    except OSError:
        from google.colab import files

        print("Download the raw file, then upload it below / 请下载该源码的原始文件并在下方上传：")
        print(f"https://github.com/{repository}/blob/{source_commit}/{source_path}")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Select one source file / 请选择一个源码文件")
        source = next(iter(uploaded.values()))

    if sha256(source).hexdigest() != expected_digest:
        raise ValueError("Source checksum mismatch; use the linked commit / 源码校验未通过，请使用指定提交的文件")
    with TemporaryDirectory() as temp:
        verified_source = Path(temp) / "clean.py"
        verified_source.write_bytes(source)
        run_audit = run_path(str(verified_source))["run_audit"]
    print("Source commit / 源码提交：", source_commit)

## 3. Generate and read the audit table

The function verifies the workbook checksum, then applies these rules in order: keep the first of any exact duplicates across all eight columns, remove rows without a customer ID, remove invoices beginning with `C`, remove nonpositive quantities, and remove nonpositive unit prices.

In the CSV, `step` and `rule` identify the position and rule. `raw_flagged` counts all matching rows in the original data, including overlaps between rules. `rows_before` counts the rows entering the step. `rows_removed` counts matching rows still present at that step. Subtracting it from `rows_before` gives `rows_after`, the number passed to the next step. A rule can flag original rows and remove zero additional rows if earlier rules already removed all its matches. Running the cell writes the audit CSV to the path shown above.

## 3. 生成并解读审计表

函数先校验工作簿哈希，再依次处理五项规则。八列完全相同的记录只保留第一条、剔除缺失客户编号的行、剔除发票号以 `C` 开头的行、剔除数量非正的行、剔除单价非正的行。

在 CSV 中，`step` 和 `rule` 分别是执行顺序和规则名称。`raw_flagged` 统计原始数据中符合该规则的全部行，规则之间可以重叠。`rows_before` 是进入该步骤的行数。`rows_removed` 是这一步实际移除的行数。二者相减得到 `rows_after`，即留给下一步的行数。如果所有命中行都已被前面的规则移除，本步仍可以有原始命中，但新增移除为零。运行下方单元格会把审计表写入上方显示的路径。

In [ ]:
audit = run_audit(workbook, output)
print("Saved / 已保存：", output)
print(audit.to_string(index=False))

## Reconcile the row counts

For the verified workbook, the five entries in `rows_removed` sum to 149,217. The last `rows_after` is 392,692. Together they account for all 541,909 original rows. In particular, 10,624 rows have nonpositive quantities in the original sheet. All were already excluded by the time the quantity rule runs, so its additional removals are zero. The remaining rows are transaction lines before merchandise filtering and basket aggregation.

## 核对行数

对于已校验的工作簿，五个步骤的 `rows_removed` 合计 149,217 行，最后一行的 `rows_after` 为 392,692。两者相加正好是原表的 541,909 行。例如，原表有 10,624 行数量非正，执行数量规则时这些记录已全部被此前的规则剔除，因此这一步新增移除 0 行。剩余的 392,692 行是交易明细，后续还要筛选有效商品并聚合为购物篮。